# Libraries

In [ ]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Qwen3-4B"
MODEL_PRETRAINED = "Qwen/Qwen3-4B"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=12,  max_gen_toks=1024),

    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]

# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    "enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] gsm8k


2026-04-05:18:37:45 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-05:18:37:51 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-04-05:18:37:53 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 285876.76 examples/s]
2026-04-05:18:39:01 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 8
2026-04-05:18:39:01 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 1319/1319 [3:06:44<00:00,  8.49s/it] 
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'Qwen/Qwen3-4B', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 8, batch_size: 4
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|---------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     8|exact_match|↑  |0.8415|±  |0.0101|
|         |       |strict-match    |     8|exact_match|↑  |0.0387|±  |0.0053|

Done

[INSTRUCTION_FOLLOWING] ifeval


2026-04-05:21:46:13 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-05:21:46:18 INFO     [_cli.run:376] Selected Tasks: ['ifeval']
2026-04-05:21:46:20 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating train split: 100%|██████████| 541/541 [00:00<00:00, 46917.51 examples/s]
2026-04-05:21:46:34 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 541/541 [2:05:24<00:00, 13.91s/it]  
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'Qwen/Qwen3-4B', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 0, batch_size: 4
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.8669|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.8429|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.8059|±  |0.0170|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.7745|±  |0.0180|

Done

[REASONING] arc_challenge


...

hf ({'pretrained': 'Qwen/Qwen3-4B', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------|------:|------|-----:|--------|---|-----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  |0.3660|±  |0.0141|
|             |       |none  |     0|acc_norm|↑  |0.3737|±  |0.0141|

Done

[PERPLEXITY] wikitext


2026-04-05:23:54:45 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
Loading weights: 100%|██████████| 398/398 [00:04<00:00, 84.56it/s]
2026-04-05:23:54:59 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-05:23:54:59 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-04-05:23:54:59 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-05:23:54:59 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-04-05:23:54:59 WARNING  [api.task:729] [Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
2026-04-05:23:54:59 WARNING  [api.task:741] [T

hf ({'pretrained': 'Qwen/Qwen3-4B', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 1
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.7685|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.7035|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |17.2643|±  |   N/A|

Done

[KNOWLEDGE] mmlu


2026-04-06:00:06:00 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-06:00:06:00 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:00:06:05 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2615.56 examples/s]
2026-04-06:00:07:39 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
2026-04-06:00:07:39 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 5
2026-04-06:00:07:39 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 5
2026-04-06:00:07:39 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 5
2026-04-06:00:07:39 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
2026-04-06:00:07:39 WARNING  [evaluato

hf ({'pretrained': 'Qwen/Qwen3-4B', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.6705|±  |0.0038|
| - humanities                          |      2|none  |      |acc   |↑  |0.5765|±  |0.0069|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.6587|±  |0.0424|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.8000|±  |0.0312|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.7892|±  |0.0286|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.8312|±  |0.0244|
|  - international_law                  |      1|none  |     5|acc  

# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)